# Hands-on 38 (Bab 15): Proyek Akhir End-to-End

Buku Referensi: *IoT Terintegrasi dengan Machine Learning & Data Analytics* (Onno W. Purbo), Bab 15.

Capstone: simulator -> MQTT -> subscriber Python (validasi payload, preprocessing,
threshold baseline + Isolation Forest) -> storage -> alert. Struktur skrip mengikuti
buku: `scripts/publishers/`, `scripts/inference/`, `scripts/subscribers/`.

In [1]:
import os
from pathlib import Path

while not (Path.cwd() / "scripts" / "mqtt").is_dir():
    os.chdir("..")
print("[INFO] Working directory:", Path.cwd())

[INFO] Working directory: /home/sandbox/iot-ml-book


## 1. Latih Model Baseline Isolation Forest

In [2]:
import subprocess, sys
r = subprocess.run([sys.executable, "scripts/inference/train_capstone_iforest.py"], capture_output=True, text=True)
print(r.stdout)

[SUKSES] Model disimpan ke: outputs/models/capstone_iforest.joblib



## 2. Jalankan Sistem (broker + subscriber+alert + listener + publisher)

In [3]:
import time

broker = subprocess.Popen([sys.executable, "scripts/mqtt/run_broker.py"],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(3)
try:
    sub = subprocess.Popen(
        [sys.executable, "scripts/subscribers/capstone_subscriber_with_alert.py",
         "--max-messages", "15", "--timeout", "45"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    listener = subprocess.Popen(
        [sys.executable, "scripts/mqtt/subscribe_sensor.py",
         "--topic", "capstone/iotnode01/alert", "--max-messages", "1", "--timeout", "45"],
        stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    time.sleep(1)
    subprocess.run([sys.executable, "scripts/publishers/capstone_publisher.py",
                    "--count", "15", "--interval", "0.3", "--seed", "7"],
                   capture_output=True, text=True)
    out_sub, _ = sub.communicate()
    print("--- Subscriber + alert ---")
    print(out_sub)
    out_listener, _ = listener.communicate()
    print("--- Listener topik alert ---")
    print(out_listener[:500])
finally:
    broker.terminate()

--- Subscriber + alert ---
[INFO] Connected to broker.
[INFO] Subscribed to capstone/iotnode01/telemetry
[2026-10-22T08:13:55] -> anomaly=1, threshold=0
[PERINGATAN] ALERT: {"timestamp": "2026-10-22T08:13:55", "device_id": "iotnode01", "alert_reason": "isolation_forest", "temperature_c": 27.24, "humidity_pct": 45.28, "power_watt": 197.17, "current_amp": 3.93}
[2026-10-22T08:13:55] -> anomaly=1, threshold=0
[PERINGATAN] ALERT: {"timestamp": "2026-10-22T08:13:55", "device_id": "iotnode01", "alert_reason": "isolation_forest", "temperature_c": 27.66, "humidity_pct": 42.03, "power_watt": 171.34, "current_amp": 3.72}
[2026-10-22T08:13:55] -> anomaly=1, threshold=0
[PERINGATAN] ALERT: {"timestamp": "2026-10-22T08:13:55", "device_id": "iotnode01", "alert_reason": "isolation_forest", "temperature_c": 24.7, "humidity_pct": 43.17, "power_watt": 156.41, "current_amp": 8.46}
[2026-10-22T08:13:55] -> anomaly=1, threshold=0
[PERINGATAN] ALERT: {"timestamp": "2026-10-22T08:13:55", "device_id": "iotnod

## 3. Verifikasi Storage

In [4]:
import pandas as pd
mon = pd.read_csv("outputs/reports/capstone_monitoring.csv")
alerts = pd.read_csv("outputs/reports/capstone_alerts.csv")
print(mon.head(8).to_string(index=False))
print(f"\nAnomali: {mon['iforest_anomaly'].sum()}, Threshold: {mon['threshold_alert'].sum()}, Alert: {len(alerts)}")

          timestamp device_id  temperature_c  humidity_pct  power_watt  current_amp  iforest_anomaly  threshold_alert
2026-10-22T08:13:55 iotnode01          27.24         45.28  197.170000       3.9300                1                0
2026-10-22T08:13:55 iotnode01          27.66         42.03  171.340000       3.7200                1                0
2026-10-22T08:13:55 iotnode01          24.70         43.17  156.410000       8.4600                1                0
2026-10-22T08:13:55 iotnode01          26.23         61.96  250.590000       6.9600                1                0
2026-10-22T08:13:56 iotnode01          33.76         41.63  234.520000       5.2400                1                0
2026-10-22T08:13:56 iotnode01          25.18         50.80  226.900000       4.5800                1                0
2026-10-22T08:13:56 iotnode01          30.39         53.03  279.185871       7.9212                1                0
2026-10-22T08:13:57 iotnode01          28.28         51.

## Penutup

Sistem capstone berjalan ujung ke ujung dan alert sampai ke topik terpisah. Satu
temuan operasional penting: baseline training buku (distribusi normal sempit) tidak
representatif terhadap rentang publisher (uniform lebar), sehingga hampir semua pesan
ditandai anomali - persis pelajaran Bab 13: data training harus mewakili kondisi
deployment, dan monitoring drift diperlukan. Untuk deployment penuh, aktifkan
`--influx` dan susun dashboard Grafana (panel suhu, kelembaban, daya, arus,
iforest_anomaly, threshold_alert, histori, dan jumlah alert per jam).